In [8]:
#1.Spark Session initialized
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("StreamingAssignment") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

print(f"Spark Session initialized. Version: {spark.version}")

Spark Session initialized. Version: 3.5.1


In [9]:
#2.Schema contract defined
#3.ReadStream 
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

user_schema = StructType([
    StructField("DEST_COUNTRY_NAME", StringType(), True),
    StructField("ORIGIN_COUNTRY_NAME", StringType(), True),
    StructField("count", IntegerType(), True)
])

input_directory = "/workspace/input_data"
streaming_df = spark.readStream \
    .format("csv") \
    .option("header", "true") \
    .schema(user_schema) \
    .load(input_directory)

print(f"Is Streaming DataFrame active? {streaming_df.isStreaming}")

Is Streaming DataFrame active? True


In [10]:
#4.Data aggregation
from pyspark.sql.functions import sum

aggregated_df = streaming_df.groupBy("ORIGIN_COUNTRY_NAME", "DEST_COUNTRY_NAME") \
    .agg(sum("count").alias("total_count"))

print("Grouped by Pair & Total Count calculated")

Grouped by Pair & Total Count calculated


In [ ]:
#5.Write Stream & Update Mode
query = aggregated_df.writeStream \
    .format("console") \
    .outputMode("update") \
    .option("truncate", "false") \
    .start()

print("query started")

26/06/14 20:21:31 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-cbe19979-eb86-4db3-8181-77b0ce025df9. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/06/14 20:21:31 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


query started


-------------------------------------------
Batch: 0
-------------------------------------------
+-------------------+-----------------+-----------+
|ORIGIN_COUNTRY_NAME|DEST_COUNTRY_NAME|total_count|
+-------------------+-----------------+-----------+
|Saint Martin       |United States    |3          |
|Guinea             |United States    |2          |
|Romania            |United States    |43         |
|Croatia            |United States    |6          |
|Ireland            |United States    |1685       |
|India              |United States    |391        |
|United States      |Egypt            |89         |
|United States      |Equatorial Guinea|2          |
|Niger              |United States    |1          |
|Singapore          |United States    |97         |
|Grenada            |United States    |308        |
|United States      |Costa Rica       |3119       |
|United States      |Senegal          |192        |
|United States      |Moldova          |1          |
|Marshall Islands  